<a href="https://colab.research.google.com/github/2003UJAN/Laryngeal-Cancer-Few-Shot-Learning-XAI/blob/main/Laryngeal_Cancer_XAI_VGGNET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pytorch_metric_learning

In [ ]:
!pip install grad-cam

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from pytorch_metric_learning import losses
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt
from torchvision.models import vgg16
import zipfile
import torch.nn as nn
import torch.optim as optim
from torchvision.models import vgg16 # Import the vgg16 function
import os

In [ ]:
zip_path = 'archive.zip'
extract_path = '.'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
dataset_path = os.path.join(extract_path, 'laryngeal dataset', 'laryngeal dataset')

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((100, 100)),
    transforms.ToTensor()
])

In [ ]:
def load_dataset(root_path, transform):
    return datasets.ImageFolder(root=root_path, transform=transform)

In [ ]:
fold1_path = os.path.join(dataset_path, 'FOLD 1')
fold2_path = os.path.join(dataset_path, 'FOLD 2')
fold3_path = os.path.join(dataset_path, 'FOLD 3')

In [ ]:
train_data = load_dataset(fold1_path, transform)
val_data = load_dataset(fold2_path, transform)
test_data = load_dataset(fold3_path, transform)

In [ ]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
class VGGNetPrototypicalNetwork(nn.Module):
    def __init__(self, embedding_dim, num_classes):
        super(VGGNetPrototypicalNetwork, self).__init__()
        self.vgg = vgg16(pretrained=True)
        self.vgg.features[0] = nn.Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

        # Flatten the output of the features
        self.flatten = nn.Flatten()

        # Replace the entire classifier with a new sequential model
        # Calculate the correct input size for the first linear layer
        num_features = self.vgg.features(torch.randn(1, 3, 100, 100)).view(1, -1).size(1) # Assuming image size is 100x100
        self.vgg.classifier = nn.Sequential(
            nn.Linear(num_features, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, embedding_dim)
        )

        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        features = self.vgg.features(x)
        features = self.flatten(features) # Flatten the features
        embeddings = self.vgg.classifier(features)
        logits = self.classifier(embeddings)
        return logits

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
embedding_dim = 64
num_classes = len(train_data.classes)
model = VGGNetPrototypicalNetwork(embedding_dim, num_classes).to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.7)
criterion = nn.CrossEntropyLoss()

In [ ]:
def compute_accuracy(loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            predictions = torch.argmax(logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return correct / total

for epoch in range(45):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()
    train_acc = compute_accuracy(train_loader)
    val_acc = compute_accuracy(val_loader)

    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}, '
          f'Train Accuracy: {train_acc*100:.2f}%, Val Accuracy: {val_acc*100:.2f}%')

test_acc = compute_accuracy(test_loader)
print(f'Test Accuracy: {test_acc*100:.2f}%')

Epoch 1, Loss: 1.6980069449969701, Train Accuracy: 26.59%, Val Accuracy: 25.45%
Epoch 2, Loss: 1.403859053339277, Train Accuracy: 33.86%, Val Accuracy: 29.09%
Epoch 3, Loss: 1.3578368382794517, Train Accuracy: 29.55%, Val Accuracy: 29.32%
Epoch 4, Loss: 1.2323427157742637, Train Accuracy: 28.18%, Val Accuracy: 26.14%
Epoch 5, Loss: 1.1740769914218359, Train Accuracy: 59.77%, Val Accuracy: 53.86%
Epoch 6, Loss: 0.999642184802464, Train Accuracy: 57.73%, Val Accuracy: 54.09%
Epoch 7, Loss: 1.0017195386545998, Train Accuracy: 51.82%, Val Accuracy: 50.91%
Epoch 8, Loss: 0.9335651525429317, Train Accuracy: 38.41%, Val Accuracy: 41.14%
Epoch 9, Loss: 1.0065594528402602, Train Accuracy: 57.50%, Val Accuracy: 55.00%
Epoch 10, Loss: 1.1568149328231812, Train Accuracy: 46.59%, Val Accuracy: 42.50%
Epoch 11, Loss: 0.9574462473392487, Train Accuracy: 64.55%, Val Accuracy: 51.82%
Epoch 12, Loss: 0.9732240736484528, Train Accuracy: 55.45%, Val Accuracy: 47.27%
Epoch 13, Loss: 0.8732606938907078, Tra

In [ ]:
def generate_gradcam(model, image, target_layer):
    model.eval()
    if image.shape[0] != model.vgg.features[0].weight.shape[1]:
        image = image.repeat(model.vgg.features[0].weight.shape[1], 1, 1)
    cam = GradCAM(model=model, target_layers=[target_layer])
    grayscale_cam = cam(input_tensor=image.unsqueeze(0))[0, :]
    image = image.permute(1, 2, 0).cpu().numpy()
    cam_image = show_cam_on_image(image, grayscale_cam, use_rgb=True)
    return cam_image

In [ ]:
test_iter = iter(test_loader)
images, labels = next(test_iter)
image = images[0].to(device)
target_layer = model.resnet.layer4[-1]
cam_image = generate_gradcam(model, image, target_layer)

In [ ]:
plt.imshow(cam_image)
plt.title('Grad-CAM')
plt.show()